# B · Motores de tablas

**📄 PÁGINA · Motores de tablas** · https://rayalucaria.org/fdd_o26/python/stack/tablas/

**📓 NOTEBOOK · b_tablas.ipynb · celdas B.0–B.10**

La página tiene la tabla de cerca, las figuras y los resultados. Este notebook corre cada comparación: arriba de cada celda, qué compara y qué mirar; el código ya escrito; abajo, «Deberías ver» con la salida de referencia.

¿Te perdiste? Corre B.0 y salta a la celda que vamos.

- **La pregunta de todo el notebook:** el total vendido (`cantidad * precio`) por tienda, sin cantidades inválidas (vacías o ≤ 0), de mayor a menor.
- **Polars es el centro.** pandas aparece sólo para comparar: lo vas a leer en código ajeno y en lo que escribe la IA.
- Corre en orden con Shift+Enter. Arriba a la derecha debe decir `.venv`.
- Cada «Deberías ver» se corrió con `N = 1_000_000 · Intel i7-7700HQ (4 núcleos, 8 hilos) · 31 GB`. Tus números serán otros; **lo que se mantiene es el patrón**.
- Antes de entregar: *Clear All Outputs* y guarda.

## B.0 · Preparación

Corre esta celda primero, y otra vez cuando te pierdas: es **idempotente** (correrla dos veces deja lo mismo). Deja listo todo lo que usan las demás.

- **`N`**: el tamaño de los datos. Deja **una sola** línea `N = …` sin `#`. B.0 te recomienda uno según tu RAM, pero **no cambia `N` sola**.
- `datos.genera(N)` — escribe `datos/ventas_{N}.csv` y `.parquet` la primera vez; después los reusa. Si ya corriste A.0 con el mismo `N`, no escribe nada.
- `CSV` — la ruta **relativa** a `stack/`, la carpeta donde corre el kernel. Así los planes de B.4 imprimen `datos/ventas_1000000.csv` y no tu ruta entera.
- `N_FILAS = min(N, 200_000)` — el tope para las carreras que recorren **objetos de Python** fila por fila (B.1): con más tardan minutos y el patrón no cambia.
- `pl.read_csv` — Polars lee el archivo **entero, ahora** (modo *eager*, el inmediato). `ventas` es la tabla de Polars de todo el notebook.
- `pd.read_csv` — lo mismo en pandas: `ventas_pd`. Las dos están en la memoria del kernel; por eso, al terminar, *Restart*.

In [ ]:
import subprocess
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import polars as pl

import datos

# Elige el tamaño según tu máquina: deja UNA línea sin #.
N = 1_000_000        # por defecto: cualquier laptop
# N = 100_000        # 8 GB de RAM o menos, o si una celda tarda más de 30 s
# N = 10_000_000     # 16 GB o más; hasta ~3 GB en disco en datos/

datos.recomienda_n()
datos.genera(N)
CSV = Path("datos") / f"ventas_{N}.csv"  # relativa a stack/, donde corre el kernel
N_FILAS = min(N, 200_000)  # tope para las carreras con objetos de Python

ventas = pl.read_csv(CSV)  # Polars: la tabla de este notebook
ventas_pd = pd.read_csv(CSV)  # pandas: sólo para comparar
print(
    f"polars {pl.__version__} · duckdb {duckdb.__version__} · pandas {pd.__version__}"
)
print(
    f"ventas: {ventas.height:_} filas × {ventas.width} columnas · N_FILAS = {N_FILAS:_}"
)

**Deberías ver:** (recortado: tus rutas empiezan distinto)

```text
Tienes 31 GB de RAM → te recomendamos N = 1_000_000
Con más de 16 GB también puedes probar N = 10_000_000.
…/stack/datos/ventas_1000000.csv  1_000_000 filas  32.4 MB
…/stack/datos/ventas_1000000.parquet  1_000_000 filas  6.0 MB
polars 2.0.0 · duckdb 1.5.6 · pandas 3.0.6
ventas: 1_000_000 filas × 5 columnas · N_FILAS = 200_000
```

`N = 1_000_000 · Intel i7-7700HQ (4 núcleos, 8 hilos) · 31 GB`

- Si dice `polars 1.…`, el kernel no es el del `.venv` de esta carpeta: *Select Kernel* → *Jupyter Kernel…* → **fdd · stack (3.14)**.
- El mismo millón de filas ocupa 32.4 MB en CSV y 6.0 MB en Parquet: eso es la página Archivos y memoria (https://rayalucaria.org/fdd_o26/python/stack/archivos/).

## B.1 · La escalera: dónde se hace la multiplicación

**Qué compara:** la misma cuenta, `cantidad * precio` sumado sobre todas las filas, hecha en cinco lugares.

| Opción | Quién recorre las filas |
|---|---|
| `for` | El intérprete: ejecuta el bytecode del cuerpo una vez por fila (https://rayalucaria.org/fdd_o26/python/por-dentro/) |
| expresión generadora | El intérprete también: `sum(c * p for …)` es el mismo `for` escrito en una línea |
| NumPy | Un ciclo compilado en C sobre el arreglo entero: **una** llamada desde Python |
| Polars, expresión | Un ciclo compilado en Rust sobre la columna entera: **una** llamada desde Python |
| pandas `.apply(…, axis=1)` | El intérprete, y peor: arma un objeto `Series` por fila y llama tu `lambda` con él |

**Fuera del intérprete** quiere decir eso: el trabajo por fila lo hace código compilado (C, Rust), no bytecode de Python.

**La particularidad de esta medición:**

- El `for` y la generadora recorren **listas de Python** de `N_FILAS` filas; NumPy y Polars, las `N` filas.
- **`.apply` se mide una sola vez (`veces=1`) sobre `M = min(N, 100_000)` filas** y se extrapola: con un millón tardaría ~8 s.
- Para comparar tamaños distintos, todo se reporta en **`seg_por_millon`**: segundos por cada millón de filas.
- Convertir las columnas a listas y arreglos (`to_list`, `to_numpy`) queda **fuera** del reloj: se mide sólo la cuenta.
- `fill_null(0)` — en una lista, un vacío es `None`, y `None * 12.5` truena. Con `0` la suma no cambia. La expresión de Polars no lo necesita: `null * 12.5` es `null`, y `sum` se salta los `null`.

**Qué mirar:** la columna `veces_vs_mejor`.

In [ ]:
M = min(N, 100_000)  # .apply se mide sobre M filas y se extrapola
cantidades = ventas["cantidad"].fill_null(0).head(N_FILAS).to_list()
precios = ventas["precio"].head(N_FILAS).to_list()
cantidad_np = ventas["cantidad"].fill_null(0).to_numpy()
precio_np = ventas["precio"].to_numpy()


def con_for():
    total = 0.0
    for c, p in zip(cantidades, precios, strict=True):
        total += c * p
    return total


def por_millon(segundos, filas):
    return segundos / filas * 1_000_000  # segundos por cada millón de filas


carrera = {
    "for": por_millon(datos.cronometra(con_for), N_FILAS),
    "expresión generadora": por_millon(
        datos.cronometra(
            lambda: sum(c * p for c, p in zip(cantidades, precios, strict=True))
        ),
        N_FILAS,
    ),
    "NumPy": por_millon(datos.cronometra(lambda: (cantidad_np * precio_np).sum()), N),
    "Polars, expresión": por_millon(
        datos.cronometra(
            lambda: ventas.select((pl.col("cantidad") * pl.col("precio")).sum())
        ),
        N,
    ),
    "pandas .apply": por_millon(
        datos.cronometra(
            lambda: (
                ventas_pd.head(M)
                .apply(lambda fila: fila["cantidad"] * fila["precio"], axis=1)
                .sum()
            ),
            veces=1,
        ),
        M,
    ),
}
datos.tabla(carrera, columna="seg_por_millon").with_columns(
    pl.col("seg_por_millon").round(4)
)

**Deberías ver:**

```text
shape: (5, 3)
┌──────────────────────┬────────────────┬────────────────┐
│ opcion               ┆ seg_por_millon ┆ veces_vs_mejor │
│ ---                  ┆ ---            ┆ ---            │
│ str                  ┆ f64            ┆ f64            │
╞══════════════════════╪════════════════╪════════════════╡
│ NumPy                ┆ 0.0021         ┆ 1.0            │
│ Polars, expresión    ┆ 0.0036         ┆ 1.7            │
│ expresión generadora ┆ 0.1329         ┆ 62.9           │
│ for                  ┆ 0.142          ┆ 67.2           │
│ pandas .apply        ┆ 8.312          ┆ 3933.6         │
└──────────────────────┴────────────────┴────────────────┘
```

`N = 1_000_000 · Intel i7-7700HQ (4 núcleos, 8 hilos) · 31 GB`

- **Dentro del intérprete, decenas de veces más lento** (aquí, de 20× a 70× entre corridas); con `.apply`, **miles de veces**.
- NumPy y Polars **casi empatan**: los dos corren fuera del intérprete. El `for` y la generadora también casi empatan; su orden cambia entre corridas.
- `.apply(axis=1)` es el más lento aunque «sea pandas»: es un `for` de Python escondido, más un objeto por fila.

**📄 PÁGINA · Motores de tablas · sección «Fuera del intérprete es rápido»**

## B.2 · La pregunta en pandas, Polars y DuckDB

**Qué compara:** la notación. La misma pregunta, escrita tres veces, y la comprobación de que las tres dan lo mismo.

| Paso | pandas | Polars | DuckDB |
|---|---|---|---|
| Filtrar | `df[df["cantidad"] > 0]` | `.filter(pl.col("cantidad") > 0)` | `WHERE cantidad > 0` |
| La cuenta | multiplicar dos `Series` | una expresión **dentro de `.agg`** | `SUM(cantidad * precio)` |
| Agrupar | `.groupby(filas["tienda"])` | `.group_by("tienda")` | `GROUP BY tienda` |
| Ordenar | `.sort_values(ascending=False)` | `.sort("total", descending=True)` | `ORDER BY total DESC` |
| Qué regresa | `Series` con `tienda` de **índice** | `DataFrame`, sin índice | una **relación** |

- El **índice** de pandas son las etiquetas de fila que guarda aparte de las columnas; aquí, los nombres de tienda. Polars no tiene índice: `tienda` es una columna más.
- `pl.col("cantidad")` es una **expresión**: describe una columna y lo que le haces, sin calcular nada hasta que un método (`filter`, `agg`) la usa.
- Una **relación** de DuckDB (`DuckDBPyRelation`) es la consulta lista, **sin traer el resultado a Python**. `.pl()` la ejecuta y la trae como `DataFrame` de Polars; `.df()`, como pandas; `.fetchall()`, como lista de tuplas.
- `read_csv('…')` dentro del SQL: DuckDB lee el archivo él mismo, en la consulta.

**La particularidad: los vacíos.** pandas leyó los vacíos como `NaN`, y `NaN > 0` es `False`. En Polars, `null > 0` es `null`, y `filter` descarta lo que no es `True`. En SQL, `NULL > 0` es `NULL`, y `WHERE` lo descarta. Tres reglas distintas, el mismo resultado.

`np.allclose` compara los totales con tolerancia: sumar floats en otro orden cambia el último decimal.

In [ ]:
# pandas: filtra con una máscara, multiplica columnas, agrupa una Serie.
filas = ventas_pd[ventas_pd["cantidad"] > 0]
r_pd = (
    (filas["cantidad"] * filas["precio"])
    .groupby(filas["tienda"])
    .sum()
    .sort_values(ascending=False)
)

# Polars: una cadena de métodos con expresiones pl.col(...).
r_pl = (
    ventas.filter(pl.col("cantidad") > 0)
    .group_by("tienda")
    .agg((pl.col("cantidad") * pl.col("precio")).sum().alias("total"))
    .sort("total", descending=True)
)

# DuckDB: SQL sobre el archivo; regresa una relación, todavía sin datos.
r_db = duckdb.sql(f"""
    SELECT tienda, SUM(cantidad * precio) AS total
    FROM read_csv('{CSV}')
    WHERE cantidad > 0
    GROUP BY tienda
    ORDER BY total DESC
""")

tipos = [type(r).__name__ for r in (r_pd, r_pl, r_db)]
print("pandas → {} | Polars → {} | DuckDB → {}".format(*tipos))
mismo_orden = (
    r_pd.index.to_list() == r_pl["tienda"].to_list() == r_db.pl()["tienda"].to_list()
)
mismos_totales = np.allclose(r_pd.to_numpy(), r_pl["total"]) and np.allclose(
    r_pl["total"], r_db.pl()["total"]
)
print("¿mismo orden?", mismo_orden, "| ¿mismos totales?", mismos_totales)
r_pl

**Deberías ver:**

```text
pandas → Series | Polars → DataFrame | DuckDB → DuckDBPyRelation
¿mismo orden? True | ¿mismos totales? True
shape: (8, 2)
┌──────────┬──────────┐
│ tienda   ┆ total    │
│ ---      ┆ ---      │
│ str      ┆ f64      │
╞══════════╪══════════╡
│ Poniente ┆ 1.7056e8 │
│ Santa Fe ┆ 1.7050e8 │
│ Sur      ┆ 1.7048e8 │
│ Oriente  ┆ 1.7047e8 │
│ Centro   ┆ 1.7024e8 │
│ Polanco  ┆ 1.6995e8 │
│ Coyoacán ┆ 1.6934e8 │
│ Norte    ┆ 1.6907e8 │
└──────────┴──────────┘
```

`N = 1_000_000 · Intel i7-7700HQ (4 núcleos, 8 hilos) · 31 GB`

- Tres tipos distintos, la misma respuesta: `True` y `True`.
- `1.7056e8` es 170 560 000: Polars abrevia los números grandes al mostrarlos; el valor guardado está completo.

**📄 PÁGINA · Motores de tablas · sección «Tres notaciones, la misma respuesta»**

## B.3 · Carrera de motores: tiempo y memoria del proceso

**Qué compara:** la pregunta completa, **leyendo el CSV**, en cuatro opciones: pandas, Polars *eager* (`read_csv`), Polars *lazy* (`scan_csv`, B.4) y DuckDB. Cada opción corre en `mide.py`, en un **proceso nuevo**, 3 veces; queda la corrida con el mejor tiempo.

El **RSS** (*resident set size*) es la memoria que el sistema le dio al proceso entero; `htop` la muestra en la columna RES. `rss_pico_mb` es el máximo que llegó a tener.

**Tres trampas de medir, y cómo las esquiva `mide.py`:**

| Trampa | Lo que hace `mide.py` |
|---|---|
| `tracemalloc` sólo ve la memoria que pide Python; Polars reserva la suya en Rust y DuckDB en C++ | Mide el RSS del proceso, que lo incluye todo |
| El pico de RSS sólo sube: en un mismo proceso, la segunda opción heredaría el pico de la primera | Un proceso nuevo por opción |
| El `import` cuesta tiempo y memoria | El reloj arranca **después** del import; `rss_base_mb` es la memoria con la librería ya cargada; `delta_mb = rss_pico_mb − rss_base_mb` |

- `subprocess.run([sys.executable, "mide.py", …])` — corre `mide.py` en otro proceso. `sys.executable` es el Python de tu `.venv`: el proceso nuevo usa tus mismas librerías.
- `check=True` — si `mide.py` falla, la celda truena aquí y no arma una tabla con basura.
- Con datos chicos domina lo fijo (arrancar, leer el encabezado): con `N = 100_000` los cuatro casi empatan.
- **Tarda ~10 s** con `N = 1_000_000`: son 12 procesos. Con `N = 10_000_000`, ~1.5 min (el orden de memoria es el mismo).

**Qué mirar:** la tabla viene ordenada por `rss_pico_mb`. Compara el pico, no `delta_mb`: pandas trae más de 100 MB sólo de importarse.

In [ ]:
def mide(escenario):
    """Corre mide.py en un proceso NUEVO y regresa su línea como dict."""
    salida = subprocess.run(
        [sys.executable, "mide.py", escenario, str(N)],
        capture_output=True,
        text=True,
        check=True,
    )
    opcion, segundos, base, pico, delta = salida.stdout.strip().split("\t")
    return {
        "opcion": opcion,
        "segundos": float(segundos),
        "rss_base_mb": float(base),
        "rss_pico_mb": float(pico),
        "delta_mb": float(delta),
    }


filas = []
for escenario in ["pandas", "polars", "polars-lazy", "duckdb"]:
    corridas = [mide(escenario) for _ in range(3)]  # 3 procesos por opción
    filas.append(min(corridas, key=lambda fila: fila["segundos"]))  # el mejor tiempo

motores = pl.DataFrame(filas).sort("rss_pico_mb", descending=True)
motores.with_columns(
    (pl.col("segundos") / pl.col("segundos").min()).round(1).alias("veces_vs_mejor")
)

**Deberías ver:**

```text
shape: (4, 6)
┌─────────────┬──────────┬─────────────┬─────────────┬──────────┬────────────────┐
│ opcion      ┆ segundos ┆ rss_base_mb ┆ rss_pico_mb ┆ delta_mb ┆ veces_vs_mejor │
│ ---         ┆ ---      ┆ ---         ┆ ---         ┆ ---      ┆ ---            │
│ str         ┆ f64      ┆ f64         ┆ f64         ┆ f64      ┆ f64            │
╞═════════════╪══════════╪═════════════╪═════════════╪══════════╪════════════════╡
│ pandas      ┆ 0.965    ┆ 105.6       ┆ 303.5       ┆ 197.9    ┆ 13.6           │
│ polars      ┆ 0.112    ┆ 46.4        ┆ 253.4       ┆ 207.0    ┆ 1.6            │
│ polars-lazy ┆ 0.071    ┆ 46.5        ┆ 146.1       ┆ 99.7     ┆ 1.0            │
│ duckdb      ┆ 0.227    ┆ 54.1        ┆ 104.4       ┆ 50.4     ┆ 3.2            │
└─────────────┴──────────┴─────────────┴─────────────┴──────────┴────────────────┘
```

`N = 1_000_000 · Intel i7-7700HQ (4 núcleos, 8 hilos) · 31 GB`

- **Memoria, el eje estable:** pandas > Polars eager > Polars lazy > DuckDB, en todas las corridas.
- **Tiempo, con `N = 1_000_000`:** pandas es el último (de 6× a 14× en distintas corridas). Polars, lazy y DuckDB **casi empatan**: su orden cambia entre corridas.
- En `delta_mb`, pandas y Polars eager casi empatan: pandas ya traía 105 MB de base. Por eso se compara el pico.
- Lazy usa la mitad que eager con el mismo Polars: B.4 muestra por qué.

**📄 PÁGINA · Motores de tablas · sección «La memoria es el eje estable»**

## B.4 · Polars lazy: el plan antes que los datos

**Eager** (`pl.read_csv`) ejecuta cada método en el momento, sobre la tabla entera. **Lazy** (`pl.scan_csv`) no lee nada: regresa un `LazyFrame`, que es un **plan**, la lista de pasos por hacer. Cada método que encadenas agrega un paso. `.collect()` optimiza el plan y lo ejecuta.

- `consulta.explain(optimized=False)` — el plan **como lo escribiste**.
- `consulta.explain()` — el plan **que corre**, después del optimizador.
- Los planes se leen **de abajo hacia arriba**: abajo está `Csv SCAN` (leer el archivo); cada nivel recibe lo de abajo (`FROM`).
- **Pushdown** es bajar un paso hasta la lectura: un filtro (*predicate pushdown*) o una selección de columnas (*projection pushdown*). El lector descarta filas y columnas **mientras lee**, y lo que no sirve nunca llega a la memoria.

**Qué mirar:** dos líneas cambian entre los dos planes. Encuéntralas antes de leer «Deberías ver».

In [ ]:
plan = pl.scan_csv(CSV)  # no lee el archivo: arma un plan
consulta = (
    plan.filter(pl.col("cantidad") > 0)
    .group_by("tienda")
    .agg((pl.col("cantidad") * pl.col("precio")).sum().alias("total"))
    .sort("total", descending=True)
)
print("── el plan como lo escribiste: explain(optimized=False) ──")
print(consulta.explain(optimized=False))
print()
print("── el plan que corre: explain() ──")
print(consulta.explain())
consulta.collect()  # ahora sí: lee, filtra, agrupa

**Deberías ver:**

```text
── el plan como lo escribiste: explain(optimized=False) ──
SORT BY [descending: [true]] [col("total")]
  AGGREGATE[maintain_order: false]
    [(col("cantidad").cast(Float64) * col("precio")).sum().alias("total")] BY [col("tienda")]
    FROM
    FILTER col("cantidad") > 0
    FROM
      Csv SCAN [datos/ventas_1000000.csv]
      PROJECT */5 COLUMNS
      ESTIMATED ROWS: 926208

── el plan que corre: explain() ──
SORT BY [descending: [true]] [col("total")]
  AGGREGATE[maintain_order: false]
    [(col("cantidad").cast(Float64) * col("precio")).sum().alias("total")] BY [col("tienda")]
    FROM
    Csv SCAN [datos/ventas_1000000.csv]
    PROJECT 3/5 COLUMNS
    SELECTION: col("cantidad") > 0
    ESTIMATED ROWS: 926208
```

(y la tabla de 8 tiendas de B.2, con los mismos totales)

`N = 1_000_000 · Intel i7-7700HQ (4 núcleos, 8 hilos) · 31 GB`

**Las dos líneas que cambian:**

| Sin optimizar | Optimizado | Qué significa |
|---|---|---|
| `PROJECT */5 COLUMNS` | `PROJECT 3/5 COLUMNS` | Lee sólo `tienda`, `cantidad` y `precio`: `fecha` y `producto` nunca se leen (*projection pushdown*) |
| `FILTER col("cantidad") > 0`, un paso aparte arriba del `Csv SCAN` | `SELECTION: col("cantidad") > 0`, **dentro** del `Csv SCAN` | El filtro se aplica al leer: las filas inválidas nunca llegan a memoria (*predicate pushdown*) |

- Esto explica B.3: lazy llegó a 146 MB y eager a 253 MB con la misma librería. Eager leyó las 5 columnas y todas las filas antes de filtrar.
- `.cast(Float64)` lo agregó Polars en los dos planes: `cantidad` es entero y `precio` decimal; para multiplicarlos, convierte `cantidad`.
- `ESTIMATED ROWS` es una estimación que Polars saca de una muestra del archivo, no un conteo.

## B.5 · Lazy, parte 2: cuándo truena, qué motor corre, escribir sin juntar

**1 · El error espera.** Armar el plan no revisa los nombres de columna. El error sale cuando algo **necesita el plan**: `.collect()`, `.explain()`, `.collect_schema()`, o **mostrar el `LazyFrame`** como última línea de una celda (el notebook dibuja el plan). Por eso `mal` se asigna sin mostrarse. Es lo mismo que viste en la 9.2 (https://rayalucaria.org/fdd_o26/python/por-dentro/que-es-python/): el error espera a que su línea corra.

**2 · Dos motores para el mismo plan.**

| Motor | Qué hace |
|---|---|
| `engine="in-memory"` | Ejecuta cada paso sobre la tabla entera: necesita todo en memoria a la vez |
| `engine="streaming"` | Ejecuta el plan **por partes** (lotes de filas): lee un lote, lo filtra, lo suma, y pasa al siguiente. Puede procesar archivos más grandes que tu RAM |

**En Polars 2.0, `.collect()` sin argumento usa `engine="auto"`, que es streaming** si no configuraste otra cosa. Compruébalo con `help(pl.LazyFrame.collect)`. En Polars 1.x, `"auto"` era in-memory: el código y las respuestas de la IA de antes de octubre de 2026 asumen eso.

**3 · Escribir sin juntar.** `.sink_parquet(ruta)` ejecuta el plan y escribe el resultado directo a un archivo Parquet, por partes, **sin armar la tabla en memoria**. Aquí escribe `datos/validas_{N}.parquet`, dentro de `datos/` (que git ignora).

- `pl.exceptions.ColumnNotFoundError` — la excepción de una columna que no existe; se atrapa para que la celda siga.
- `str(error).splitlines()[0]` — la primera línea del mensaje; el mensaje entero trae también el plan hasta donde falló.
- `pl.scan_parquet(destino).select(pl.len()).collect().item()` — cuenta las filas del archivo escrito sin leer sus columnas.

In [ ]:
# 1 · El error espera: «cantidda» está mal escrita.
mal = pl.scan_csv(CSV).filter(pl.col("cantidda") > 0)
print("Asignar no truena: nadie ha pedido el plan todavía.")
try:
    mal.collect()
except pl.exceptions.ColumnNotFoundError as error:
    print("collect() →", type(error).__name__ + ":", str(error).splitlines()[0])

# 2 · Dos motores para el mismo plan.
validas = pl.scan_csv(CSV).filter(pl.col("cantidad") > 0)
print(
    datos.tabla(
        {
            'collect(engine="in-memory")': datos.cronometra(
                lambda: validas.collect(engine="in-memory")
            ),
            'collect(engine="streaming")': datos.cronometra(
                lambda: validas.collect(engine="streaming")
            ),
        }
    )
)

# 3 · Del CSV a Parquet sin juntar la tabla en memoria.
destino = Path("datos") / f"validas_{N}.parquet"
validas.sink_parquet(destino)
print(
    f"{destino}  {pl.scan_parquet(destino).select(pl.len()).collect().item():_} filas"
)

**Deberías ver:**

```text
Asignar no truena: nadie ha pedido el plan todavía.
collect() → ColumnNotFoundError: unable to find column "cantidda"; valid columns: ["fecha", "tienda", "producto", "cantidad", "precio"]
shape: (2, 3)
┌─────────────────────────────┬──────────┬────────────────┐
│ opcion                      ┆ segundos ┆ veces_vs_mejor │
│ ---                         ┆ ---      ┆ ---            │
│ str                         ┆ f64      ┆ f64            │
╞═════════════════════════════╪══════════╪════════════════╡
│ collect(engine="streaming") ┆ 0.072025 ┆ 1.0            │
│ collect(engine="in-memory") ┆ 0.080078 ┆ 1.1            │
└─────────────────────────────┴──────────┴────────────────┘
datos/validas_1000000.parquet  980_042 filas
```

`N = 1_000_000 · Intel i7-7700HQ (4 núcleos, 8 hilos) · 31 GB`

- El error nombra la columna mala y lista las válidas; el mensaje completo agrega `Did you mean "cantidad"?`.
- Los dos motores **casi empatan** en tiempo con un millón de filas. Su diferencia es la memoria: streaming nunca junta la tabla entera.
- 980 042 filas: el millón menos las ~2 % con cantidad vacía o ≤ 0.

**📄 PÁGINA · Motores de tablas · sección «Lazy: el plan antes que los datos»**

## B.6 · El tipo decide la memoria

**Qué compara:** la misma columna `tienda` (8 textos distintos repetidos en todas las filas) guardada con cinco tipos, y `cantidad` como entero de 64 y de 32 bits.

| Tipo | Cómo guarda cada fila |
|---|---|
| pandas `object` | Un puntero a un objeto `str` de Python, cada uno con su encabezado |
| pandas `str` | El tipo de texto **por omisión desde pandas 3.0**: los bytes en un arreglo de Arrow |
| Polars `String` | Los bytes del texto, en buffers de Rust |
| Polars `Categorical` | Cada texto distinto **una vez**, y por fila sólo un número que lo señala |
| Polars `Enum` | Igual que `Categorical`, pero la lista de valores se fija **antes** (`pl.Enum(datos.TIENDAS)`): un valor fuera de la lista es un error al convertir |
| `Int64` / `Int32` | 8 / 4 bytes por fila. `Int32` llega hasta 2 147 483 647 |

- `memory_usage(deep=True)` — pandas cuenta también el contenido de cada objeto, no sólo los punteros.
- `estimated_size()` — el tamaño de los buffers de Polars, en bytes.
- **`sys.getsizeof` mide sólo el objeto de Python, no los buffers en Rust a los que apunta.** Un `DataFrame` de Polars es un objeto de Python de 48 bytes que señala datos guardados en Rust.

In [ ]:
MB = 1_000_000
tienda_pd = ventas_pd["tienda"]
tienda = ventas["tienda"]
memoria = {
    "pandas object": tienda_pd.astype(object).memory_usage(deep=True, index=False) / MB,
    "pandas str": tienda_pd.memory_usage(deep=True, index=False) / MB,
    "Polars String": tienda.estimated_size() / MB,
    "Polars Categorical": tienda.cast(pl.Categorical).estimated_size() / MB,
    "Polars Enum": tienda.cast(pl.Enum(datos.TIENDAS)).estimated_size() / MB,
}
print(datos.tabla(memoria, columna="mb").with_columns(pl.col("mb").round(2)))

print("cantidad Int64:", round(ventas["cantidad"].estimated_size() / MB, 2), "MB")
print(
    "cantidad Int32:",
    round(ventas["cantidad"].cast(pl.Int32).estimated_size() / MB, 2),
    "MB",
)
print(
    "sys.getsizeof(ventas):",
    sys.getsizeof(ventas),
    "bytes  ← sólo el objeto de Python, no los buffers en Rust",
)
print("ventas.estimated_size():", round(ventas.estimated_size() / MB, 1), "MB")

**Deberías ver:**

```text
shape: (5, 3)
┌────────────────────┬───────┬────────────────┐
│ opcion             ┆ mb    ┆ veces_vs_mejor │
│ ---                ┆ ---   ┆ ---            │
│ str                ┆ f64   ┆ f64            │
╞════════════════════╪═══════╪════════════════╡
│ Polars Enum        ┆ 1.0   ┆ 1.0            │
│ Polars Categorical ┆ 4.0   ┆ 4.0            │
│ Polars String      ┆ 6.62  ┆ 6.6            │
│ pandas str         ┆ 14.62 ┆ 14.6           │
│ pandas object      ┆ 57.49 ┆ 57.5           │
└────────────────────┴───────┴────────────────┘
cantidad Int64: 8.13 MB
cantidad Int32: 4.13 MB
sys.getsizeof(ventas): 48 bytes  ← sólo el objeto de Python, no los buffers en Rust
ventas.estimated_size(): 36.7 MB
```

`N = 1_000_000 · Intel i7-7700HQ (4 núcleos, 8 hilos) · 31 GB`

- **La misma columna, de 1 MB a 57 MB según el tipo.** `object` era el tipo de texto por omisión antes de pandas 3.0: es el que asume el código viejo, y el que escribe la IA.
- `Enum` gana porque cada fila guarda un número pequeño y la lista de 8 tiendas se guarda una vez.
- `Int32` es la mitad de `Int64`: úsalo cuando sabes que los valores caben.
- `getsizeof` dice 48 bytes de una tabla de 36.7 MB.

## B.7 · Un vacío en una columna de enteros

**Qué compara:** qué hace cada librería con `[3, None, 5]`, y con la columna `cantidad` del CSV, que trae ~1 % de vacíos.

- Los enteros de pandas son de NumPy (`int64`), que **no tienen un valor «vacío»**. Un solo `None` obliga a pandas a convertir la columna entera a `float64` y a guardar el vacío como `NaN` (*Not a Number*, un valor especial de los floats).
- Polars guarda, junto a cada columna, una **máscara de validez**: un bit por fila que dice si hay valor o `null`. La columna sigue siendo `Int64`.
- Un `float64` guarda enteros exactos sólo hasta 2⁵³ = 9 007 199 254 740 992. Arriba de eso, dos enteros vecinos pueden quedar iguales: un id o un monto en centavos cambia sin aviso.
- pandas tiene un entero con vacíos, `"Int64"` con mayúscula, pero hay que pedirlo: `pd.read_csv(…, dtype={"cantidad": "Int64"})`.

In [ ]:
print(pd.Series([3, None, 5], name="pandas"))
print(pl.Series("polars", [3, None, 5]))

grande = 2**53 + 1  # 9007199254740993: un float64 ya no lo distingue de su vecino
print(
    "pandas:",
    pd.Series([grande, None]).iloc[0],
    "| Polars:",
    pl.Series([grande, None])[0],
)

print(
    "cantidad del CSV · pandas:",
    ventas_pd["cantidad"].dtype,
    "| Polars:",
    ventas["cantidad"].dtype,
)

**Deberías ver:**

```text
0    3.0
1    NaN
2    5.0
Name: pandas, dtype: float64
shape: (3,)
Series: 'polars' [i64]
[
	3
	null
	5
]
pandas: 9007199254740992.0 | Polars: 9007199254740993
cantidad del CSV · pandas: float64 | Polars: Int64
```

`N = 1_000_000 · Intel i7-7700HQ (4 núcleos, 8 hilos) · 31 GB`

- En pandas, el `3` se volvió `3.0`: **un vacío cambió el tipo de toda la columna.**
- En Polars, `3` sigue siendo `3` y el vacío es `null`.
- `9007199254740993` llegó a pandas como `…992.0`: perdió una unidad sin error.

**📄 PÁGINA · Motores de tablas · sección «El tipo decide la memoria»**

## B.8 · Compartir vs copiar

**Arrow** es un formato estándar para guardar tablas **por columnas en memoria**. Polars, DuckDB y PyArrow lo usan por dentro; cuando dos librerías lo comparten, una puede leer los buffers de la otra **sin copiarlos**.

**Qué compara:** la misma pregunta en SQL, con DuckDB, sobre dos fuentes ya cargadas en el kernel; y el costo de pasar la tabla de Polars a pandas y de regreso.

- `FROM {tabla}` — `sql.format(tabla="ventas")` escribe ahí el nombre de **una variable de Python**. DuckDB no tiene ninguna tabla `ventas`: busca una variable con ese nombre y la lee.
- `DuckDB sobre Polars` — lee los buffers Arrow de `ventas`.
- `DuckDB sobre pandas` — la misma consulta sobre `ventas_pd`, cuyas columnas DuckDB tiene que convertir antes de consultarlas.
- `ida_y_vuelta` — `pl.from_pandas(ventas.to_pandas())`: copiar la tabla a pandas y de regreso, **sin calcular nada**. No va en la tabla (no es una forma de contestar); se imprime aparte, como fracción de lo que tarda la consulta sobre Polars.

**Qué mirar:** cuánto cuesta la ida y vuelta comparada con contestar la pregunta entera. Tarda ~2 s con `N = 1_000_000`; ~30 s con `N = 10_000_000`.

In [ ]:
sql = """
    SELECT tienda, SUM(cantidad * precio) AS total
    FROM {tabla} WHERE cantidad > 0
    GROUP BY tienda ORDER BY total DESC
"""
# «ventas» es la variable de Polars de B.0: DuckDB la encuentra por su nombre.
print(duckdb.sql(sql.format(tabla="ventas")).pl().head(3))

consulta = {
    "DuckDB sobre Polars": datos.cronometra(
        lambda: duckdb.sql(sql.format(tabla="ventas")).pl()
    ),
    "DuckDB sobre pandas": datos.cronometra(
        lambda: duckdb.sql(sql.format(tabla="ventas_pd")).pl()
    ),
}
print(datos.tabla(consulta))

# La ida y vuelta no contesta nada: se compara contra la consulta sobre Polars.
ida_y_vuelta = datos.cronometra(lambda: pl.from_pandas(ventas.to_pandas()))
fraccion = ida_y_vuelta / consulta["DuckDB sobre Polars"]
print(f"ida y vuelta por pandas: {ida_y_vuelta:.3f} s", end=" = ")
print(f"{fraccion:.0%} de contestar la pregunta con DuckDB")

**Deberías ver:**

```text
shape: (3, 2)
┌──────────┬──────────┐
│ tienda   ┆ total    │
│ ---      ┆ ---      │
│ str      ┆ f64      │
╞══════════╪══════════╡
│ Poniente ┆ 1.7056e8 │
│ Santa Fe ┆ 1.7050e8 │
│ Sur      ┆ 1.7048e8 │
└──────────┴──────────┘
shape: (2, 3)
┌─────────────────────┬──────────┬────────────────┐
│ opcion              ┆ segundos ┆ veces_vs_mejor │
│ ---                 ┆ ---      ┆ ---            │
│ str                 ┆ f64      ┆ f64            │
╞═════════════════════╪══════════╪════════════════╡
│ DuckDB sobre Polars ┆ 0.106347 ┆ 1.0            │
│ DuckDB sobre pandas ┆ 0.776119 ┆ 7.3            │
└─────────────────────┴──────────┴────────────────┘
ida y vuelta por pandas: 0.078 s = 73% de contestar la pregunta con DuckDB
```

`N = 1_000_000 · Intel i7-7700HQ (4 núcleos, 8 hilos) · 31 GB`

- **La misma consulta es varias veces más lenta sobre pandas que sobre Polars** (de 5× a 8× entre corridas).
- La ida y vuelta por pandas cuesta **casi tanto como contestar la pregunta entera** con DuckDB (de 60 % a 95 % entre corridas), y no contesta nada.
- En código: si ya tienes un `DataFrame` de Polars, pásalo tal cual a DuckDB; no lo conviertas a pandas en medio.

## B.9 · Núcleos: el motor en Rust usa todos

**Antes de correr:** abre `htop` en una terminal aparte, junto a VS Code, como en la 9.2 (https://rayalucaria.org/fdd_o26/python/por-dentro/el-gil/).

En la 9.2 viste que un proceso de Python ejecuta código Python en **un** núcleo a la vez: el GIL. Polars ejecuta su trabajo en Rust, **fuera del GIL**, con un grupo de hilos del tamaño de tus CPU lógicas (`pl.thread_pool_size()` te dice cuántos).

- `polars` — la pregunta con Polars eager, todos los hilos.
- `polars-1-hilo` — lo mismo, pero `mide.py` fija `POLARS_MAX_THREADS=1` **antes** de importar Polars: Polars lee esa variable al cargarse, y después ya no la cambia.
- `ciclo(escenario)` — repite `mide.py` durante ~10 s y se queda con el mejor tiempo. **La celda dura ~20 s a propósito**: es el tiempo para mirar `htop`.

**Qué mirar en `htop`:** con `polars`, varias barras de CPU suben a la vez; con `polars-1-hilo`, una sola.

In [ ]:
import time


def ciclo(escenario, segundos=10):
    """Repite mide.py durante ~10 s; regresa el mejor tiempo y cuántas corridas hizo."""
    tiempos = []
    fin = time.perf_counter() + segundos
    while time.perf_counter() < fin:
        salida = subprocess.run(
            [sys.executable, "mide.py", escenario, str(N)],
            capture_output=True,
            text=True,
            check=True,
        )
        tiempos.append(float(salida.stdout.split("\t")[1]))
    print(f"{escenario}: {len(tiempos)} corridas")
    return min(tiempos)


datos.tabla({"polars": ciclo("polars"), "polars-1-hilo": ciclo("polars-1-hilo")})

**Deberías ver:**

```text
polars: 22 corridas
polars-1-hilo: 14 corridas
shape: (2, 3)
┌───────────────┬──────────┬────────────────┐
│ opcion        ┆ segundos ┆ veces_vs_mejor │
│ ---           ┆ ---      ┆ ---            │
│ str           ┆ f64      ┆ f64            │
╞═══════════════╪══════════╪════════════════╡
│ polars        ┆ 0.119    ┆ 1.0            │
│ polars-1-hilo ┆ 0.319    ┆ 2.7            │
└───────────────┴──────────┴────────────────┘
```

`N = 1_000_000 · Intel i7-7700HQ (4 núcleos, 8 hilos) · 31 GB`

- Con 8 CPU lógicas, Polars fue ~2.7× más rápido que con un hilo. No llega a 8×: los 8 hilos comparten 4 núcleos físicos, y no todo el trabajo se reparte.
- **El GIL no frena a Polars**: el trabajo pesado nunca pasa por el intérprete.

**📄 PÁGINA · Motores de tablas · sección «Compartir, no copiar»**

## B.10 · «La IA te dio esto»

Pídele a una IA «el total vendido por tienda, ignorando cantidades inválidas, en Python» y es probable que recibas algo como `total_ia`: pandas, una copia, y `.apply(lambda fila: …, axis=1)` con un `if` adentro. **Es correcto y se lee bien.** El problema es dónde corre: una llamada de Python por fila (B.1).

- `df.copy()` — copia la tabla entera para no modificar la original.
- `.apply(lambda fila: … if fila["cantidad"] > 0 else 0, axis=1)` — recorre las filas en Python; el `if` decide fila por fila.
- `total_polars` — la misma pregunta como expresiones: el `if` se volvió `filter`.

Las dos se miden sobre las mismas `M = min(N, 100_000)` filas; `.apply` una sola vez (`veces=1`). Antes de comparar tiempos, se comprueba que den lo mismo.

In [ ]:
M = min(N, 100_000)
muestra_pd, muestra = ventas_pd.head(M), ventas.head(M)


def total_ia(df):  # lo que escribió la IA
    df = df.copy()
    df["total"] = df.apply(
        lambda fila: fila["cantidad"] * fila["precio"] if fila["cantidad"] > 0 else 0,
        axis=1,
    )
    return df.groupby("tienda")["total"].sum().sort_values(ascending=False)


def total_polars(df):  # la misma pregunta, como expresión de Polars
    return (
        df.filter(pl.col("cantidad") > 0)
        .group_by("tienda")
        .agg((pl.col("cantidad") * pl.col("precio")).sum().alias("total"))
        .sort("total", descending=True)
    )


print(
    "¿mismo resultado?",
    np.allclose(total_ia(muestra_pd).to_numpy(), total_polars(muestra)["total"]),
)
datos.tabla(
    {
        "pandas .apply (la IA)": datos.cronometra(
            lambda: total_ia(muestra_pd), veces=1
        ),
        "Polars, expresión": datos.cronometra(lambda: total_polars(muestra)),
    }
)

**Deberías ver:**

```text
¿mismo resultado? True
shape: (2, 3)
┌───────────────────────┬──────────┬────────────────┐
│ opcion                ┆ segundos ┆ veces_vs_mejor │
│ ---                   ┆ ---      ┆ ---            │
│ str                   ┆ f64      ┆ f64            │
╞═══════════════════════╪══════════╪════════════════╡
│ Polars, expresión     ┆ 0.003132 ┆ 1.0            │
│ pandas .apply (la IA) ┆ 0.97191  ┆ 310.3          │
└───────────────────────┴──────────┴────────────────┘
```

`N = 1_000_000 · Intel i7-7700HQ (4 núcleos, 8 hilos) · 31 GB`

- Mismo resultado, **cientos de veces más lento**, con sólo 100 000 filas.
- Al revisar código de IA, `.apply(lambda fila:` con `axis=1` es un `for` de Python escondido. Pregunta qué expresión lo reemplaza.

**📄 Regresa a la página Motores de tablas.** Antes, *Restart* o *Shutdown* de este kernel: libera la memoria para el siguiente notebook.